In [2]:
!pip install transformers accelerate sentencepiece -q


In [3]:
from huggingface_hub import login
from google.colab import userdata

token = userdata.get('HF_TOKEN')

login(token)

In [4]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
import torch

model_id = "google/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

In [5]:
system_prompt = """
You are VitaEdge, an offline AI health triage assistant for rural healthcare workers.

Your responsibilities:
- Analyze patient symptoms carefully
- Estimate urgency level
- Suggest medically sensible possible conditions
- Explain your reasoning step-by-step
- Recommend safe next actions
- Encourage referral for severe symptoms

Safety Rules:
- Never guarantee diagnosis
- Never pretend to replace a doctor
- Recommend referral for severe or uncertain symptoms
- Avoid dangerous medical advice

For every patient case, respond using EXACTLY this format:

1. Urgency Level
2. Possible Conditions
3. Step-by-Step Reasoning
   - Symptoms observed
   - Clinical rule applied
   - Conclusion reached
4. Recommended Action
5. Confidence Score (0-100)
6. Referral Needed (Yes/No)

Keep explanations simple and understandable for community health workers.
"""

In [8]:
def ask_vitaedge(patient_case):

    prompt = system_prompt + "\n\nPatient Case:\n" + patient_case

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.2,
        do_sample=True,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove prompt from output
    response = response.replace(prompt, "").strip()

    print(response)

In [10]:
ask_vitaedge("""

Man age 60
Severe chest pain
Sweating
Pain spreading to left arm

""")

**1. Urgency Level:** High

**2. Possible Conditions:**

* Heart attack
* Stroke
* Pneumonia
* Angina pectoris

**3. Step-by-Step Reasoning:**

**Symptoms observed:** Severe chest pain, sweating, pain spreading to left arm

**Clinical rule applied:** Rule out heart attack based on absence of classic symptoms like shortness of breath, nausea, and vomiting

**Conclusion reached:** Heart attack is highly suspected

**4. Recommended Action:**

* Call 911 immediately

**5. Confidence Score:** 95%

**6. Referral Needed:** Yes


In [ ]:
for i, example in enumerate(training_examples, 1):

    print(f"\nExample {i}")
    print("-" * 50)

    print("Instruction:")
    print(example["instruction"])

    print("\nInput:")
    print(example["input"])

    print("\nExpected Output:")
    print(example["output"])

In [ ]:
import json

with open("vitaedge_dataset.json", "w") as f:
    json.dump(training_examples, f, indent=4)

print("Dataset saved successfully!")

In [ ]:
!pip install datasets -q

In [ ]:
from datasets import load_dataset

dataset = load_dataset("medalpaca/medical_meadow_medqa")

In [ ]:
print(dataset)

In [ ]:
for i in range(3):

    print(f"\n================ EXAMPLE {i+1} ================\n")

    print("INSTRUCTION:")
    print(dataset["train"][i]["instruction"])

    print("\nINPUT:")
    print(dataset["train"][i]["input"])

    print("\nOUTPUT:")
    print(dataset["train"][i]["output"])

In [ ]:
vitaedge_dataset = []

In [ ]:
def add_example(input_text,
                urgency,
                condition,
                reasoning,
                action,
                referral):

    example = {

        "instruction":
        "Analyze the patient case and provide triage guidance.",

        "input": input_text,

        "output": {

            "urgency": urgency,

            "possible_condition": condition,

            "reasoning": reasoning,

            "action": action,

            "refer_to_doctor": referral
        }
    }

    vitaedge_dataset.append(example)

In [ ]:
add_example(

    input_text=
    "Child age 3 with fever 39C, chest indrawing, cough, and difficulty breathing.",

    urgency="CRITICAL",

    condition="Possible pneumonia",

    reasoning=
    "High fever with chest indrawing and breathing difficulty suggests severe respiratory infection.",

    action=
    "Immediate hospital referral required.",

    referral=True
)

In [ ]:
add_example(

    input_text=
    "Adult age 22 with runny nose, sneezing, mild sore throat, and no fever.",

    urgency="LOW",

    condition="Possible mild cold or allergies",

    reasoning=
    "Symptoms are mild with no signs of severe infection.",

    action=
    "Home care, hydration, and rest recommended.",

    referral=False
)

In [ ]:
add_example(

    input_text=
    "Man age 60 with severe chest pain, sweating, and pain spreading to left arm.",

    urgency="CRITICAL",

    condition="Possible cardiac emergency",

    reasoning=
    "Chest pain with sweating and arm pain may indicate heart attack.",

    action=
    "Immediate emergency medical attention required.",

    referral=True
)

In [ ]:
# Example 4 — Dehydration

add_example(

    input_text=
    "Child age 5 with diarrhea, vomiting, dry mouth, and sunken eyes.",

    urgency="HIGH",

    condition="Possible dehydration",

    reasoning=
    "Vomiting, diarrhea, dry mouth, and sunken eyes are signs of dehydration.",

    action=
    "Start oral rehydration immediately and seek medical evaluation soon.",

    referral=True
)

In [ ]:
# Example 5 — Malaria-like Symptoms

add_example(

    input_text=
    "Adult age 30 with high fever, chills, sweating, and body pain in a mosquito-prone area.",

    urgency="HIGH",

    condition="Possible malaria",

    reasoning=
    "High fever with chills and sweating in a mosquito-prone area may indicate malaria.",

    action=
    "Medical evaluation and malaria testing recommended.",

    referral=True
)

In [ ]:
# Example 6 — Mild Fever

add_example(

    input_text=
    "Child age 8 with mild fever 37.8C and headache but eating normally.",

    urgency="LOW",

    condition="Possible mild viral infection",

    reasoning=
    "Mild fever with stable appetite and no severe symptoms suggests a minor illness.",

    action=
    "Rest, hydration, and monitoring recommended.",

    referral=False
)

In [ ]:
# Example 7 — Severe Asthma Attack

add_example(

    input_text=
    "Teenager age 15 with wheezing, chest tightness, and severe difficulty breathing.",

    urgency="CRITICAL",

    condition="Possible severe asthma attack",

    reasoning=
    "Wheezing and severe breathing difficulty suggest serious airway obstruction.",

    action=
    "Immediate emergency medical care required.",

    referral=True
)

In [ ]:
# Example 8 — Malnutrition

add_example(

    input_text=
    "Child age 4 with visible weight loss, swollen belly, weakness, and poor appetite.",

    urgency="HIGH",

    condition="Possible severe malnutrition",

    reasoning=
    "Weight loss, weakness, and abdominal swelling are concerning signs of malnutrition.",

    action=
    "Urgent nutritional and medical assessment recommended.",

    referral=True
)

In [ ]:
# Example 9 — Minor Injury

add_example(

    input_text=
    "Adult age 26 with small cut on hand and mild bleeding after kitchen accident.",

    urgency="LOW",

    condition="Minor superficial injury",

    reasoning=
    "Small cut with controlled bleeding and no severe symptoms is low risk.",

    action=
    "Clean wound, apply antiseptic, and monitor for infection.",

    referral=False
)

In [ ]:
# Example 10 — Severe Infection Risk

add_example(

    input_text=
    "Infant age 1 with fever 39.5C, lethargy, poor feeding, and weak crying.",

    urgency="CRITICAL",

    condition="Possible severe infection",

    reasoning=
    "High fever, lethargy, and poor feeding in an infant are dangerous warning signs.",

    action=
    "Immediate hospital referral required.",

    referral=True
)

In [ ]:
print("Total examples:", len(vitaedge_dataset))

In [ ]:
import json

with open("vitaedge_dataset.json", "w") as f:

    json.dump(vitaedge_dataset, f, indent=4)

print("Dataset saved successfully!")

In [ ]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    vitaedge_dataset,
    test_size=0.2,
    random_state=42
)

print("Training examples:", len(train_data))
print("Validation examples:", len(val_data))

In [ ]:
import json

with open("vitaedge_train.json", "w") as f:
    json.dump(train_data, f, indent=4)

with open("vitaedge_val.json", "w") as f:
    json.dump(val_data, f, indent=4)

print("Train and validation datasets saved!")

In [1]:
!pip uninstall -y transformers huggingface_hub peft trl accelerate bitsandbytes unsloth -q

!pip install unsloth transformers accelerate peft trl bitsandbytes -q

In [ ]:
import unsloth

from unsloth import FastLanguageModel

import torch

In [ ]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(

    model_name = "google/gemma-2b-it",

    max_seq_length = max_seq_length,

    dtype = torch.float16,

    load_in_4bit = True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(

    model,

    r = 16,

    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    lora_alpha = 16,

    lora_dropout = 0,

    bias = "none",

    use_gradient_checkpointing = True,

    random_state = 42,
)

In [ ]:
model.print_trainable_parameters()

In [ ]:
import json

with open("vitaedge_train.json", "r") as f:
    train_data = json.load(f)

with open("vitaedge_val.json", "r") as f:
    val_data = json.load(f)

print("Training examples:", len(train_data))
print("Validation examples:", len(val_data))

In [ ]:
formatted_train = []

for example in train_data:

    conversation = {
        "messages": [
            {
                "role": "user",
                "content": example["input"]
            },
            {
                "role": "assistant",
                "content": json.dumps(example["output"])
            }
        ]
    }

    formatted_train.append(conversation)

print(formatted_train[0])

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(formatted_train)

In [ ]:
def formatting_prompts_func(examples):

    texts = []

    for messages in examples["messages"]:

        user_message = messages[0]["content"]

        assistant_message = messages[1]["content"]

        formatted_text = f"""<start_of_turn>user
{user_message}
<end_of_turn>

<start_of_turn>model
{assistant_message}
<end_of_turn>
"""

        texts.append(formatted_text)

    return {"text": texts}

In [ ]:
train_dataset = train_dataset.map(
    formatting_prompts_func,
    batched=True,
)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

In [ ]:
trainer = SFTTrainer(

    model = model,

    tokenizer = tokenizer,

    train_dataset = train_dataset,

    dataset_text_field = "text",

    max_seq_length = 2048,

    dataset_num_proc = 2,

    packing = False,

    args = TrainingArguments(

        per_device_train_batch_size = 2,

        gradient_accumulation_steps = 4,

        warmup_steps = 5,

        num_train_epochs = 3,

        learning_rate = 2e-4,

        fp16 = True,

        logging_steps = 1,

        optim = "adamw_8bit",

        weight_decay = 0.01,

        lr_scheduler_type = "linear",

        seed = 42,

        output_dir = "outputs",
    ),
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("vitaedge_lora")
tokenizer.save_pretrained("vitaedge_lora")

In [25]:
from google.colab import files

uploaded = files.upload()

Saving vitaedge.png to vitaedge.png


In [27]:
# ============================================================
# INSTALL GRADIO
# ============================================================

!pip install gradio -q


# ============================================================
# IMPORTS
# ============================================================

import gradio as gr


# ============================================================
# VITAEDGE AI FUNCTION
# ============================================================

def vitaedge_app(patient_case):

    prompt = system_prompt + "\n\nPatient Case:\n" + patient_case

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        temperature=0.2,
        do_sample=True,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    response = response.replace(prompt, "").strip()

    return response


# ============================================================
# EXAMPLES
# ============================================================

examples = [

    ["""
Child age 3
Fever 39C
Difficulty breathing
Chest indrawing
Not eating properly
"""],

    ["""
Adult age 22
Runny nose
Sneezing
No fever
Eating normally
"""],

    ["""
Man age 60
Severe chest pain
Sweating
Pain spreading to left arm
"""],

    ["""
Child age 5
Vomiting
Diarrhea
Dry mouth
Sunken eyes
"""]
]


# ============================================================
# CUSTOM CSS
# ============================================================

custom_css = """

body {
    background: #eef2ff;
}

.gradio-container {
    max-width: 100% !important;
    padding: 0px !important;
    font-family: 'Segoe UI', sans-serif !important;
}

footer {
    visibility: hidden;
}

textarea {
    border-radius: 16px !important;
    border: 2px solid #a855f7 !important;
    font-size: 18px !important;
}

button {
    background: linear-gradient(90deg,#ff2e93,#4338ca) !important;
    color: white !important;
    border: none !important;
    border-radius: 16px !important;
    font-size: 26px !important;
    font-weight: 700 !important;
    height: 72px !important;
}

"""


# ============================================================
# HEADER
# ============================================================

header_html = """

<div style="
background: linear-gradient(90deg,#7e22ce,#1d4ed8,#0ea5e9);
padding:40px;
border-radius:0px 0px 25px 25px;
margin-bottom:20px;
">

<div style="
display:flex;
justify-content:space-between;
align-items:center;
flex-wrap:wrap;
gap:20px;
">

<div>

<h1 style="
font-size:100px;
font-weight:900;
color:white;
margin:0;
line-height:1;
font-family:Arial;
">
🩺 Vita<span style="color:#67e8f9;">Edge</span>
</h1>

<h2 style="
font-size:44px;
color:white;
font-weight:700;
margin-top:10px;
margin-bottom:20px;
">
Offline AI Health Triage Assistant
</h2>

<div style="
display:inline-block;
padding:14px 26px;
background:linear-gradient(90deg,#2563eb,#d946ef);
border-radius:18px;
font-size:24px;
font-weight:600;
color:white;
">
✨ AI-powered rural healthcare support using Google's Gemma model.
</div>

</div>



</div>

</div>

"""


# ============================================================
# BUILD APP
# ============================================================

with gr.Blocks(css=custom_css) as demo:

    gr.HTML(header_html)

    with gr.Row():

        # ====================================================
        # LEFT SIDE
        # ====================================================

        with gr.Column(scale=1):

                 gr.Markdown("""

# ✨ Why VitaEdge?

### 🧠 AI-powered symptom triage

### 🚨 Urgency classification

### 📋 Explainable reasoning

### 📊 Confidence scoring

### 🏥 Referral recommendations

### 🌍 Offline-first healthcare concept

""")

        # ====================================================
        # RIGHT SIDE
        # ====================================================

        with gr.Column(scale=2):

            patient_input = gr.Textbox(
                lines=8,
                label="📋 Patient Information",
                placeholder="Enter patient symptoms and details here..."
            )

            submit_btn = gr.Button("✨ Analyze Patient")

            output_box = gr.Textbox(
                lines=18,
                label="🧠 VitaEdge Analysis"
            )

            submit_btn.click(
                fn=vitaedge_app,
                inputs=patient_input,
                outputs=output_box
            )

    # ========================================================
    # EXAMPLES
    # ========================================================

    gr.Markdown("## 📚 Example Patient Cases")

    gr.Examples(
        examples=examples,
        inputs=patient_input
    )

    # ========================================================
    # DISCLAIMER
    # ========================================================

    gr.Markdown("""

<div style="
margin-top:20px;
padding:18px;
border-radius:16px;
background:linear-gradient(90deg,#ff2e93,#2563eb);
color:white;
font-size:18px;
font-weight:600;
text-align:center;
">

⚠️ VitaEdge is an educational AI healthcare prototype and not a replacement for licensed medical professionals.

</div>

""")

# ============================================================
# LAUNCH
# ============================================================

demo.launch(share=True)

/tmp/ipykernel_45412/3427995042.py:191: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3cd9c80741ad3c2e21.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
